In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path("../data") if Path("../data").exists() else Path("data")
FALTANTES = ["", "None", "nan", "NaN", "NULL", "null"]

def leer(nombre):
    return pd.read_csv(DATA / f"{nombre}.csv",
                       dtype=str,
                       keep_default_na=False,
                       encoding="utf-8-sig"
                       )
    
cat_perfil = leer("cat_perfil_riesgo")
cat_activos = leer("catalogo_activos")
cat_banca = leer("catalogo_banca")
m = leer("historico_aba_macroactivos")
u = leer("historico_aba_usd_internacional")



In [ ]:
print(DATA.resolve())
print(sorted(p.name for p in DATA.glob("*.csv")))

In [ ]:
tablas = {
    "cat_perfil_riesgo": cat_perfil,
    "catalogo_activos": cat_activos,
    "catalogo_banca": cat_banca,
    "historico_aba_macroactivos": m,
    "historico_aba_usd_internacional": u,
}
for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")
    print("  columnas:", list(df.columns))

In [ ]:
def faltantes(df):
    f = df.isin(FALTANTES).sum()
    return f[f > 0]
print(faltantes(m))
print(faltantes(u))

In [ ]:
for col in ["macroactivo", "cod_perfil_riesgo", "cod_banca", "ingestion_year", "ingestion_month"]:
    print(f"--- {col} ---")
    print(m[col].value_counts(dropna=False).head(15))
    print()

In [ ]:
raras = m[~m["macroactivo"].isin(["Renta Variable", "FICs", "Renta Fija"] + FALTANTES)]
print(len(raras))
print(raras)

In [ ]:
print(cat_perfil)
print(cat_banca)      # ¿notas algo repetido?
print(cat_activos)    # ¿hay nombres repetidos o códigos raros?

In [ ]:
aba = pd.to_numeric(m["aba"], errors="coerce")   # lo no numérico pasa a NaN
print("aba no numérico:", aba.isna().sum())
print(aba.describe())
print(m.loc[aba.isna()])                           # mira esas filas

# Extremos: lo muy pequeño o muy grande también es sospechoso
print(m.assign(aba_n=aba).sort_values("aba_n").head(5))
print(m.assign(aba_n=aba).sort_values("aba_n").tail(5))

In [ ]:
vm = pd.to_numeric(u["valor_mercado"], errors="coerce")
print("no numéricos:", vm.isna().sum(), "| negativos:", (vm < 0).sum(), "| ceros:", (vm == 0).sum())
print(u.loc[vm < 0].head())

In [ ]:
partes = m[["ingestion_year", "ingestion_month", "ingestion_day"]].apply(pd.to_numeric, errors="coerce")
partes.columns = ["year", "month", "day"]
m["fecha"] = pd.to_datetime(partes, errors="coerce")

print("fechas inválidas:", m["fecha"].isna().sum())
print("rango:", m["fecha"].min(), "->", m["fecha"].max())

In [ ]:
u["f"] = u["ingestion_year"] + "-" + u["ingestion_month"] + "-" + u["ingestion_day"]
print(u["f"].value_counts())

In [ ]:
p = u["fecha_vencimiento"].str.split("/", expand=True)
print("primer campo > 12: ", (pd.to_numeric(p[0], errors="coerce") > 12).sum())
print("segundo campo > 12:", (pd.to_numeric(p[1], errors="coerce") > 12).sum())

In [ ]:
p = u["fecha_vencimiento"].str.split("/", expand=True)
print("primer campo > 12: ", (pd.to_numeric(p[0], errors="coerce") > 12).sum())
print("segundo campo > 12:", (pd.to_numeric(p[1], errors="coerce") > 12).sum())

In [ ]:
print(u.loc[u["nombre_activo"].str.contains("BANCOLOMBIA PANAMA SA BOND", na=False),
            ["nombre_activo", "fecha_vencimiento"]].head(3))

In [ ]:
ids = m["id_sistema_cliente"]
print("IDs distintos:", ids.nunique())
print("en notación científica:", ids.str.contains("E+", regex=False).sum())
print(ids.str.len().value_counts().sort_index())
print(sorted(ids[ids.str.contains("E+", regex=False)].unique()))

In [ ]:
print("duplicados exactos macro:", m.duplicated().sum())
print("duplicados exactos usd:  ", u.duplicated().sum())

In [ ]:
print(u.groupby(["id_sistema_cliente", "f"]).size().unstack(fill_value=0))

In [ ]:
x = u[(u["id_sistema_cliente"] == "10014876058") & (u["isin"] == "IE00B7KFL990")]
print(x[["f", "nombre_activo", "cantidad", "valor_mercado"]])

In [ ]:
# ¿Los códigos de activo de los datos existen en el catálogo?
en_datos = set(m["cod_activo"]) - set(FALTANTES)
print(sorted(en_datos - set(cat_activos["cod_activo"])))

# ¿Qué clientes están en ambas tablas?
ids_m, ids_u = set(m["id_sistema_cliente"]), set(u["id_sistema_cliente"])
print("en ambas:", len(ids_m & ids_u))
print("solo en USD:", sorted(ids_u - ids_m))